# Setup & Imports

In [34]:
import sys
import numpy as np

# Add the src directory to the Python path
sys.path.append("src")

from QuantumCircuit import QuantumCircuit
from gates.registry import GateRegistry
from error_channels.default_noise import build_default_noise_model
from error_channels.ChannelRegistry import ChannelRegistry
from gates.registry import Gate

# Create Quantum Circuit

In [5]:
# Create a 2-qubit circuit
qc = QuantumCircuit(num_qubits=2, num_cbits=2, enable_metrics=True)

qc


QuantumCircuit(num_qubits=2, num_cbits=2, gates=0)

# Apply Gates (H, X, CNOT)

In [10]:
gate_reg = GateRegistry()
# Access built-in gates through registry
H = gate_reg.get("h")
X = gate_reg.get("x")
CNOT = gate_reg.get("cx")

# Apply gates
qc.add_gate(H, 0)         # Hadamard on qubit 0
qc.add_gate(X, 1)         # X on qubit 1
qc.add_gate(CNOT, [0, 1]) # CNOT: control=0, target=1

qc.ops


[('gate', <gates.registry.Gate at 0x177ce9b6dc0>, 0),
 ('gate', <gates.registry.Gate at 0x177ce49ddc0>, 1),
 ('gate', <gates.registry.Gate at 0x177ce9b6280>, [0, 1])]

# Measure Qubits

In [11]:
qc.measure(0, 0)  # measure qubit 0 into classical bit 0
qc.measure(1, 1)  # measure qubit 1 into classical bit 1

qc.ops


[('gate', <gates.registry.Gate at 0x177ce9b6dc0>, 0),
 ('gate', <gates.registry.Gate at 0x177ce49ddc0>, 1),
 ('gate', <gates.registry.Gate at 0x177ce9b6280>, [0, 1]),
 ('measure', 0, 0),
 ('measure', 1, 1)]

# Execute the Circuit

In [12]:
result_state = qc.execute(verbose=True)

print("Final state vector:")
print(result_state)

QUANTUM CIRCUIT EXECUTION RESULT

[CIRCUIT INFO]
  Qubits:       2
  Classical:    2
  Operations:   5 (3 gates, 2 measurements, 0 resets)

[CLASSICAL REGISTER]
  Bitstring: |10⟩
  c[0] = 1
  c[1] = 0

[MEASUREMENTS]
  q[0] → 1 (stored in c[0])
  q[1] → 0 (stored in c[1])

[FINAL STATE]
  |10⟩: +1.000000  (P = 1.000000)

[PERFORMANCE METRICS]
  Execution time: 0.017626s
  Peak memory:    115.92 MB
  Memory delta:   +0.22 MB


Final state vector:
{'success': True, 'state_vector': array([0.+0.j, 0.+0.j, 1.+0.j, 0.+0.j]), 'probabilities': array([0., 0., 1., 0.]), 'classical_bits': {'c[0]': 1, 'c[1]': 0, 'bitstring': '10'}, 'measurements': {'q[0]': {'outcome': 1, 'stored_in': 'c[0]'}, 'q[1]': {'outcome': 0, 'stored_in': 'c[1]'}}, 'circuit_info': {'num_qubits': 2, 'num_cbits': 2, 'num_operations': 5, 'gate_count': 3, 'measurement_count': 2, 'reset_count': 0}, 'metrics': {'execution': {'total_time_seconds': 0.017626000000291242, 'start_time': 3049.8646443, 'end_time': 3049.8822703}, 'memory'

# Measurement Results

In [14]:
print("Measurement results:")
print(qc.get_cbits().get_bits())


Measurement results:
[1 0]


# Running Circuit Multiple Times

In [31]:
from QuantumCircuit import QuantumCircuit
from gates.registry import GateRegistry

# Set up a Bell-state circuit with 2 qubits and 2 classical bits
reg = GateRegistry()
qc_shots = QuantumCircuit(
    num_qubits=2,
    num_cbits=2,
    enable_metrics=True,   # so metrics get attached to the execution result
    num_shots=1250         # default number of shots (can be overridden below)
)

# |00> -> (H on qubit 0) -> (CX 0->1) -> Bell state (|00> + |11>)/sqrt(2)
qc_shots.add_gate(H, targets=0)
qc_shots.add_gate(CNOT, targets=[0, 1])

# Measure both qubits into classical bits c[0], c[1]
qc_shots.measure(qubit=0, cbit=0)
qc_shots.measure(qubit=1, cbit=1)

# Run the circuit multiple times ("shots")
shot_results = qc_shots.run_shots()


# Return counts and probabilities
counts = shot_results["counts"]
probs  = shot_results["probabilities"]

print("Counts (from run_shots return value):")
print(counts)
print("\nProbabilities:")
print(probs)


Counts (from run_shots return value):
{'00': 640, '11': 602, '01': 3, '10': 5}

Probabilities:
{'00': 0.512, '11': 0.4816, '01': 0.0024, '10': 0.004}


# Error Channels 

Implemented quantum noise channels:
- **Bit Flip Channel**: Flips |0⟩ ↔ |1⟩ with probability p
- **Phase Flip Channel**: Applies phase flip with probability p
- **Depolarizing Channel**: General noise model
- **Custom Kraus Operators**: User-defined error channels

In [33]:
# Initialize channel registry
chan_reg = ChannelRegistry()

print("Available channels:")
print(chan_reg.list())

# Demonstrate Bit Flip Channel
print("\n" + "="*60)
print("BIT FLIP CHANNEL")
print("="*60)
bit_flip_30 = chan_reg.get_param('bit_flip').instantiate(0.3)
print(f"{bit_flip_30}")
print(f"Number of Kraus operators: {len(bit_flip_30.kraus_ops)}")

# Apply bit flip to |0⟩ state multiple times to see stochastic behavior
print("\nApplying bit flip (p=0.3) to |0⟩ state:")
psi_0 = np.array([1.0, 0.0], dtype=complex)
for i in range(5):
    result = bit_flip_30.apply_statevector(psi_0.copy())
    print(f"Trial {i+1}: {result} -> measured as |{np.argmax(np.abs(result))}⟩")

# Demonstrate Phase Flip Channel  
print("\n" + "="*60)
print("PHASE FLIP CHANNEL (Bit-Phase Flip)")
print("="*60)
phase_flip_20 = chan_reg.get_param('bit_phase_flip').instantiate(0.2)
print(f"{phase_flip_20}")

# Apply to |+⟩ state = (|0⟩ + |1⟩)/√2
print("\nApplying phase flip (p=0.2) to |+⟩ state:")
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print(f"Initial |+⟩ state: {psi_plus}")
for i in range(3):
    result = phase_flip_20.apply_statevector(psi_plus.copy())
    print(f"Trial {i+1}: {result}")

# Demonstrate Depolarizing Channel
print("\n" + "="*60)
print("DEPOLARIZING CHANNEL")
print("="*60)

depol_15 = chan_reg.get_param('depolarizing').instantiate(0.15)
print(f"{depol_15}")
# Apply to |+⟩ state = (|0⟩ + |1⟩)/√2
print("\nApplying depolarization (p=0.15) to |+⟩ state:")
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print(f"Initial |+⟩ state: {psi_plus}")
for i in range(10):
    result = depol_15.apply_statevector(psi_plus.copy())
    print(f"Trial {i+1}: {result}")

Available channels:
['amplitude_damping', 'bit_flip', 'bit_phase_flip', 'depolarizing', 'phase_damping', 'phase_flip']

BIT FLIP CHANNEL
bit_flip (1q) Channel with 2 Kraus ops
Number of Kraus operators: 2

Applying bit flip (p=0.3) to |0⟩ state:
Trial 1: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 2: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 3: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 4: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 5: [0.+0.j 1.+0.j] -> measured as |1⟩

PHASE FLIP CHANNEL (Bit-Phase Flip)
bit_phase_flip (1q) Channel with 2 Kraus ops

Applying phase flip (p=0.2) to |+⟩ state:
Initial |+⟩ state: [0.70710678+0.j 0.70710678+0.j]
Trial 1: [0.-0.70710678j 0.+0.70710678j]
Trial 2: [0.-0.70710678j 0.+0.70710678j]
Trial 3: [0.70710678+0.j 0.70710678+0.j]

DEPOLARIZING CHANNEL
depolarizing (1q) Channel with 4 Kraus ops

Applying depolarization (p=0.15) to |+⟩ state:
Initial |+⟩ state: [0.70710678+0.j 0.70710678+0.j]
Trial 1: [0.70710678+0.j 0.70710678+0.j]
Trial 2: [0.70710678+0.j 0.707

# Noise Integration with Gates

In [36]:
# # Create a noisy Hadamard gate (10% bit flip error)
# bit_flip_10 = chan_reg.get_param('bit_flip').instantiate(0.1)
# noisy_h = Gate('noisy_h', gate_reg.get('h').matrix, noise=bit_flip_10)

# print("Comparing clean vs noisy Hadamard:")
# print("\nClean Hadamard on |0⟩:")
# psi_clean = np.array([1.0, 0.0], dtype=complex)
# result_clean = gate_reg.get('h').apply(psi_clean)
# print(f"Result: {result_clean}")
# print(f"Probabilities: {np.abs(result_clean)**2}")

# print("\nNoisy Hadamard on |0⟩ (with 10% bit flip):")
# # Run multiple times to see stochastic behavior
# results = []
# for i in range(5):
#     psi_noisy = np.array([1.0, 0.0], dtype=complex)
#     result = noisy_h.apply(psi_noisy)
#     results.append(result)
#     print(f"Trial {i+1}: {result}")

# QASM Parser

In [58]:
from qasm_parser import parse_qasm_source

qasm = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];

h q[0];
x q[1];
cx q[0], q[1];
"""

instructions = parse_qasm_source(qasm)
print(instructions.items())

print("Parsed QASM instructions:")
for instr, value in instructions.items():
    if instr == "ops":
        print("Gates and their corresponding qubit arguments:")
        for dict in value:  
                print(f"\tGate {dict['name']}: {dict['qargs']}")
    else:
        print(f"{instr}: {value}")
    
        


dict_items([('n_qubits', 2), ('n_clbits', 0), ('ops', [{'name': 'h', 'qargs': [0], 'cargs': [], 'params': [], 'condition': None}, {'name': 'x', 'qargs': [1], 'cargs': [], 'params': [], 'condition': None}, {'name': 'cx', 'qargs': [0, 1], 'cargs': [], 'params': [], 'condition': None}])])
Parsed QASM instructions:
n_qubits: 2
n_clbits: 0
Gates and their corresponding qubit arguments:
	Gate h: [0]
	Gate x: [1]
	Gate cx: [0, 1]


# Pauli Simulator

## Import necessary Libraries

In [76]:
from pauli_simulator.batch_tableau import BatchTableau
from pauli_simulator.Tableau_Ver2 import Tableau
#from pauli_simulator.pauli_error_channels import NoisySimulator

## Tableau Creation

In [70]:
def hadamard_single_qubit_demo(shots, enable_metrics=True):
    outcomes = []

    for _ in range(shots):
        # 1 qubit, no classical bits needed for this demo
        t = Tableau(n=2, num_cbits=0, enable_metrics=enable_metrics)
        t.h(0)
        m = t.measure(0)
        outcomes.append(m)

    outcomes = np.array(outcomes)
    p0 = np.mean(outcomes == 0)
    p1 = np.mean(outcomes == 1)

    print(f"Shots: {shots}")
    print(f"P(0) ≈ {p0:.3f}, P(1) ≈ {p1:.3f}")


hadamard_single_qubit_demo(shots=1000)


Shots: 1000
P(0) ≈ 0.496, P(1) ≈ 0.504


In [73]:
from collections import Counter

def bell_state_demo(shots=1000):
    counts = Counter()

    for _ in range(shots):
        t = Tableau(n=2, num_cbits=0, enable_metrics=False)
        t.h(0)
        t.cx(0, 1)
        m0 = t.measure(0)
        m1 = t.measure(1)
        bitstring = f"{m0}{m1}"  # order: q0 q1
        counts[bitstring] += 1

    print("Counts:", counts)

    # Sort keys for plotting
    keys = ["00", "01", "10", "11"]
    vals = [counts.get(k, 0) for k in keys]


    # Also print empirical probabilities
    total = sum(counts.values())
    for k in keys:
        print(f"P({k}) ≈ {counts.get(k, 0)/total:.3f}")

bell_state_demo(shots=2000)



Counts: Counter({'11': 1027, '00': 973})
P(00) ≈ 0.486
P(01) ≈ 0.000
P(10) ≈ 0.000
P(11) ≈ 0.513


In [74]:
def reset_demo(trials=20):
    t = Tableau(n=1, num_cbits=1, enable_metrics=False)

    for i in range(trials):
        # Prepare |+>
        t.reset(0)         # make sure we start at |0>
        t.h(0)
        
        # Measure into classical bit c[0]
        outcome1 = t.measure(0, cbit=0)
        c_reg_before = t.get_classical_register()["bitstring"]

        # Reset qubit to |0>
        t.reset(0)

        # Measure again (should be deterministically 0)
        outcome2 = t.measure(0, cbit=0)
        c_reg_after = t.get_classical_register()["bitstring"]

        print(
            f"Trial {i:2d}: first outcome = {outcome1}, "
            f"classical before reset = {c_reg_before}, "
            f"second outcome = {outcome2}, "
            f"classical after reset = {c_reg_after}"
        )

reset_demo(trials=10)


Trial  0: first outcome = 1, classical before reset = 1, second outcome = 0, classical after reset = 0
Trial  1: first outcome = 0, classical before reset = 0, second outcome = 0, classical after reset = 0
Trial  2: first outcome = 1, classical before reset = 1, second outcome = 0, classical after reset = 0
Trial  3: first outcome = 1, classical before reset = 1, second outcome = 0, classical after reset = 0
Trial  4: first outcome = 0, classical before reset = 0, second outcome = 0, classical after reset = 0
Trial  5: first outcome = 0, classical before reset = 0, second outcome = 0, classical after reset = 0
Trial  6: first outcome = 0, classical before reset = 0, second outcome = 0, classical after reset = 0
Trial  7: first outcome = 0, classical before reset = 0, second outcome = 0, classical after reset = 0
Trial  8: first outcome = 0, classical before reset = 0, second outcome = 0, classical after reset = 0
Trial  9: first outcome = 0, classical before reset = 0, second outcome =

In [75]:
t = Tableau(n=2, num_cbits=2, enable_metrics=False)

t.h(0)
t.cx(0, 1)
t.z(1)
t.measure(0, cbit=0)
t.measure(1, cbit=1)

t.print_circuit()
print("Final classical register:", t.get_classical_register())



TABLEAU CIRCUIT: 2 qubits, 2 classical bits

Operations (5 total):

    1. H    q[0]
    2. CX   q[0], q[1]
    3. Z    q[1]
    4. MEAS q[0] -> c[0] (outcome: 0)
    5. MEAS q[1] -> c[1] (outcome: 0)

Final classical register: 00

Final classical register: {'c[0]': 0, 'c[1]': 0, 'bitstring': '00'}


In [78]:
def random_clifford_circuit(n_qubits, depth, seed=0):
    rng = np.random.default_rng(seed)
    t = Tableau(n=n_qubits, num_cbits=0, enable_metrics=True)

    single_qubit_gates = ["h", "s", "x", "y", "z"]

    for _ in range(depth):
        gate_type = rng.choice(["single", "cx", "measure"], p=[0.6, 0.3, 0.1])

        if gate_type == "single":
            q = int(rng.integers(0, n_qubits))
            g = rng.choice(single_qubit_gates)
            if g == "h":
                t.h(q)
            elif g == "s":
                t.s(q)
            elif g == "x":
                t.x(q)
            elif g == "y":
                t.y(q)
            elif g == "z":
                t.z(q)

        elif gate_type == "cx":
            c = int(rng.integers(0, n_qubits))
            t_q = int(rng.integers(0, n_qubits))
            if c != t_q:
                t.cx(c, t_q)

        elif gate_type == "measure":
            q = int(rng.integers(0, n_qubits))
            t.measure(q)

    return t

t_rand = random_clifford_circuit(n_qubits=2, depth=100, seed=42)
t_rand.print_circuit()

metrics = t_rand.get_metrics()
t_rand.print_metrics()

metrics  # so you can also inspect the raw dict



TABLEAU CIRCUIT: 2 qubits, 0 classical bits

Operations (86 total):

    1. CX   q[1], q[0]
    2. CX   q[0], q[1]
    3. Z    q[1]
    4. X    q[1]
    5. Z    q[0]
    6. CX   q[0], q[1]
    7. S    q[0]
    8. H    q[1]
    9. CX   q[0], q[1]
   10. CX   q[1], q[0]
   11. MEAS q[0] (outcome: 0)
   12. X    q[0]
   13. Y    q[0]
   14. CX   q[1], q[0]
   15. Z    q[1]
   16. Y    q[0]
   17. Y    q[0]
   18. X    q[0]
   19. H    q[1]
   20. CX   q[1], q[0]
   21. Z    q[1]
   22. CX   q[0], q[1]
   23. Z    q[0]
   24. CX   q[0], q[1]
   25. CX   q[1], q[0]
   26. X    q[1]
   27. S    q[1]
   28. CX   q[0], q[1]
   29. H    q[0]
   30. Y    q[1]
   31. S    q[0]
   32. S    q[0]
   33. Z    q[0]
   34. X    q[0]
   35. X    q[1]
   36. CX   q[1], q[0]
   37. H    q[0]
   38. CX   q[0], q[1]
   39. Z    q[0]
   40. X    q[1]
   41. H    q[1]
   42. Y    q[0]
   43. Z    q[1]
   44. S    q[0]
   45. MEAS q[1] (outcome: 1)
   46. MEAS q[0] (outcome: 0)
   47. H    q[0]
   48. MEAS q[

{'execution': {'total_time_seconds': 0.0027336000639479607,
  'start_time': None,
  'end_time': None},
 'operations': {'gate_count': 77,
  'measurement_count': 9,
  'total_operations': 86,
  'gates_by_type': {'h': 13, 's': 10, 'cx': 18, 'x': 14, 'y': 9, 'z': 13}},
 'timing': {'gate_time_seconds': 0.0010541000519879162,
  'measurement_time_seconds': 0.0016795000119600445},
 'measurements': {'deterministic': 4,
  'probabilistic': 5,
  'outcomes': {0: 3, 1: 6}},
 'memory': {'initial_mb': 167.73046875,
  'peak_mb': 167.73046875,
  'final_mb': 167.73046875,
  'delta_mb': 0.0,
  'gate_memory': {'samples': [167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
    167.73046875,
